# Inter-Annotator Agreement(IAA) 를 계산
- 두 전문가가 얼마나 일관되게 세분류를 선택했는가?

🎯 다음 단계 전체 흐름

1️⃣ 두 파일 병합  
2️⃣ 동일 공고 기준 정렬  
3️⃣ Cohen’s Kappa 계산  
4️⃣ 불일치 케이스 추출  
5️⃣ 최종 Gold 확정 파일 생성  

## Cohen’s Kappa 계산 코드

In [6]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# ===== 파일 경로 =====
PATH_1 = "datas/processed/gold_labeling_20_1.csv"
PATH_2 = "datas/processed/gold_labeling_20_2.csv"

# ===== 로드 =====
df1 = pd.read_csv(PATH_1)
df2 = pd.read_csv(PATH_2)

# ===== 필수 컬럼 확인 =====
if "job_posting_id" not in df1.columns or "gold_role_1" not in df1.columns:
    raise ValueError("gold_labeling_20_1.csv에 job_posting_id, gold_role_1 필요")
if "job_posting_id" not in df2.columns or "gold_role_1" not in df2.columns:
    raise ValueError("gold_labeling_20_2.csv에 job_posting_id, gold_role_1 필요")

# ===== 병합 =====
merged = df1[["job_posting_id", "gold_role_1"]].merge(
    df2[["job_posting_id", "gold_role_1"]],
    on="job_posting_id",
    suffixes=("_r1", "_r2")
)

# ===== Kappa 계산 =====
kappa = cohen_kappa_score(merged["gold_role_1_r1"], merged["gold_role_1_r2"])

print("Cohen's Kappa:", round(kappa, 4))

Cohen's Kappa: 1.0


## Top-1 Accuracy 계산

- pilot_top3.csv
- gold_final_20.csv (또는 합의된 merged 파일)

In [7]:
import pandas as pd
import re

# 매핑 결과 파일 (Top-3)
PILOT_PATH = "outputs/sbert_pilot_top3.csv"
# 최종 gold 데이터
GOLD_PATH = "datas/processed/gold_final_20.csv"

# -------------------------
# 0) 유틸: 문자열/라벨 정규화
# -------------------------
def norm_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def norm_label(x):
    """띄어쓰기/특수문자 차이로 인한 불일치 방지용"""
    s = norm_str(x)
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[·\-/_,|()\[\]]", "", s)
    return s

# -------------------------
# 1) 데이터 로드
# -------------------------
pilot = pd.read_csv(PILOT_PATH, dtype={"job_posting_id": "string"}, encoding="utf-8-sig")
gold  = pd.read_csv(GOLD_PATH,  dtype={"job_posting_id": "string"}, encoding="utf-8-sig")

# -------------------------
# 2) 비교 컬럼 고정
#    gold_final  ↔  job_role_name (둘 다 한글 텍스트)
# -------------------------
gold_label_col  = "gold_final"
pilot_label_col = "job_role_name"

if gold_label_col not in gold.columns:
    raise ValueError(f"gold 파일에 '{gold_label_col}' 컬럼이 없습니다. 현재: {gold.columns.tolist()}")
if pilot_label_col not in pilot.columns:
    raise ValueError(f"pilot 파일에 '{pilot_label_col}' 컬럼이 없습니다. 현재: {pilot.columns.tolist()}")

# -------------------------
# 3) ID/라벨 정리
# -------------------------
pilot["job_posting_id"] = pilot["job_posting_id"].apply(norm_str)
gold["job_posting_id"]  = gold["job_posting_id"].apply(norm_str)
pilot["rank"]           = pilot["rank"].apply(norm_str)

pilot["pred_label_raw"] = pilot[pilot_label_col].apply(norm_str)
gold["gold_label_raw"]  = gold[gold_label_col].apply(norm_str)

pilot["pred_label"] = pilot["pred_label_raw"].apply(norm_label)
gold["gold_label"]  = gold["gold_label_raw"].apply(norm_label)

print(f"Gold label col : {gold_label_col}")
print(f"Pilot label col: {pilot_label_col}")
print(f"공통 job_posting_id 수: {len(set(gold['job_posting_id']) & set(pilot['job_posting_id']))}")

Gold label col : gold_final
Pilot label col: job_role_name
공통 job_posting_id 수: 20


In [8]:
# -------------------------
# 4) Top-1 Accuracy
# -------------------------
top1 = (
    pilot[pilot["rank"].isin(["1", "1.0"])]
    [["job_posting_id", "pred_label_raw", "pred_label"]]
    .rename(columns={"pred_label_raw": "pred_top1_raw", "pred_label": "pred_top1"})
)

eval_top1 = gold[["job_posting_id", "gold_label_raw", "gold_label"]].merge(
    top1, on="job_posting_id", how="inner"
)

if len(eval_top1) == 0:
    raise ValueError("Gold와 Pilot 간 job_posting_id 매칭이 0입니다. 두 파일의 ID를 확인하세요.")

top1_acc = (eval_top1["gold_label"] == eval_top1["pred_top1"]).mean()
top1_correct = (eval_top1["gold_label"] == eval_top1["pred_top1"]).sum()

print("========== LABEL COLUMNS ==========")
print(f"Gold label col : {gold_label_col}")
print(f"Pilot label col: {pilot_label_col}")
print()

print("========== Top-1 Accuracy ==========")
print(f"Accuracy: {top1_acc:.4f}  ({top1_correct}/{len(eval_top1)})")
print()

========== LABEL COLUMNS ==========
Gold label col : gold_final
Pilot label col: job_role_name

========== Top-1 Accuracy ==========
Accuracy: 0.1500  (3/20)



In [9]:
# -------------------------
# 5) Top-3 Accuracy
# -------------------------
top3 = (
    pilot[pilot["rank"].isin(["1", "2", "3", "1.0", "2.0", "3.0"])]
    .groupby("job_posting_id")["pred_label"]
    .apply(list)
    .reset_index()
    .rename(columns={"pred_label": "pred_top3_list"})
)

eval_top3 = gold[["job_posting_id", "gold_label"]].merge(top3, on="job_posting_id", how="inner")
eval_top3["hit_top3"] = eval_top3.apply(lambda r: r["gold_label"] in r["pred_top3_list"], axis=1)

top3_acc = eval_top3["hit_top3"].mean()
top3_correct = eval_top3["hit_top3"].sum()

print("========== Top-3 Accuracy ==========")
print(f"Accuracy: {top3_acc:.4f}  ({top3_correct}/{len(eval_top3)})")

========== Top-3 Accuracy ==========
Accuracy: 0.5500  (11/20)


## 분석 필요

2️⃣ 수치 해석 (파일럿 20개 기준)  
🔹 Top-1 = 0.15

의미:

모델이 1순위로 정확히 맞춘 비율이 15%

즉, “단일 예측 정확도”는 매우 낮음

👉 이 상태로 학술지 제출은 불가능

🔹 Top-3 = 0.55

의미:

20개 중 11개는 정답이 후보 3개 안에 있음

즉, 완전히 틀린 건 아님

“후보군은 어느 정도 의미적으로 근접”

👉 구조 자체는 완전히 무너진 건 아님

7️⃣ 박사 멘토 관점 정리

현재 상태는:

구조 붕괴 ❌

의미 근접성 존재 ⭕

분류 경계 모호 ⭕

role_text 설계 개선 필요 ⭕

## 평가(Top-1 / Top-3) 계산

(1) 라벨 컬럼을 강제로 이름 기준으로 통일해서(0 되는 문제 방지) Top-1/Top-3를 계산하고,  
(2) 실패 케이스를 자동으로 뽑아 다음 분석 단계로 넘깁니다.

In [10]:
import pandas as pd

PILOT_PATH = "outputs/sbert_pilot_top3.csv"
GOLD_PATH  = "datas/processed/gold_final_20.csv"

pilot = pd.read_csv(PILOT_PATH, dtype={"job_posting_id": "string"})
gold  = pd.read_csv(GOLD_PATH,  dtype={"job_posting_id": "string"})

# 1) ID/라벨 정리
pilot["job_posting_id"] = pilot["job_posting_id"].astype(str).str.strip()
gold["job_posting_id"]  = gold["job_posting_id"].astype(str).str.strip()

# ✅ 이름 기준 비교(강제)
pilot["job_role_name"] = pilot["job_role_name"].astype(str).str.strip()
gold["gold_final"]     = gold["gold_final"].astype(str).str.strip()

# 2) Top-1 Accuracy
top1 = (pilot[pilot["rank"] == 1][["job_posting_id", "job_role_name"]]
        .rename(columns={"job_role_name": "pred_top1"}))

eval_top1 = gold[["job_posting_id", "gold_final"]].merge(top1, on="job_posting_id", how="inner")
eval_top1["correct_top1"] = (eval_top1["gold_final"] == eval_top1["pred_top1"])

top1_acc = eval_top1["correct_top1"].mean()

# 3) Top-3 Accuracy
top3 = (pilot[pilot["rank"].isin([1,2,3])]
        .groupby("job_posting_id")["job_role_name"]
        .apply(list)
        .reset_index()
        .rename(columns={"job_role_name": "pred_top3_list"}))

eval_top3 = gold[["job_posting_id", "gold_final"]].merge(top3, on="job_posting_id", how="inner")
eval_top3["hit_top3"] = eval_top3.apply(lambda r: r["gold_final"] in r["pred_top3_list"], axis=1)

top3_acc = eval_top3["hit_top3"].mean()

print(f"[RESULT] Top-1 Accuracy: {top1_acc:.4f} ({eval_top1['correct_top1'].sum()}/{len(eval_top1)})")
print(f"[RESULT] Top-3 Accuracy: {top3_acc:.4f} ({eval_top3['hit_top3'].sum()}/{len(eval_top3)})")

# 4) 실패 케이스 저장(다음 단계: 에러 분석)
fail_top1 = eval_top1[~eval_top1["correct_top1"]].copy()
fail_top1.to_csv("datas/faild_log/fail_top1.csv", index=False, encoding="utf-8-sig")

fail_top3 = eval_top3[~eval_top3["hit_top3"]].copy()
fail_top3.to_csv("datas/faild_log/fail_top3.csv", index=False, encoding="utf-8-sig")

print("[OK] saved fail cases: datas/faild_log/fail_top1.csv, datas/faild_log/fail_top3.csv")

[RESULT] Top-1 Accuracy: 0.1500 (3/20)
[RESULT] Top-3 Accuracy: 0.5500 (11/20)
[OK] saved fail cases: datas/faild_log/fail_top1.csv, datas/faild_log/fail_top3.csv


```
D. 에러 분석(논문용 “정성 분석” 최소 세트)
임시 Gold라도 논문에서 가장 강한 부분이 여기서 나옵니다.

1) Top-1 오분류 유형 3가지로 분류

fail_top1.csv를 보면서 각 케이스에 아래 중 하나 태그만 붙이세요.

중첩형(Overlap): 정답과 예측이 같은 소분류(인공지능) 안에서 흔들림

표현형(Wording): 공고에 핵심 키워드(LLM/프롬프트/RAG)가 거의 없음

다역할형(Multi-role): 공고가 모델링+데이터+서비스구현이 섞임(Top-k가 필요)

이 태그 3개는 나중에 “성능이 낮은 이유”를 깔끔하게 설명해줍니다.

2) Top-3 미포함(fail_top3) 케이스는 별도로

Top-3에도 정답이 없다는 건 설계 문제 가능성이 큽니다:

role_text가 빈약/유사

gold 라벨이 과하게 상위 개념(생성형AI엔지니어링)으로만 찍힘

공고 text에 잡음이 많음
```

```
예시(임시 Gold 전제 포함):

“파일럿(20건) 임시 주석 기반 실험에서 Top-1은 낮았으나 Top-3에서 일정 수준의 후보 포착이 확인되었다.”

“오분류는 주로 세분류 간 의미 중첩과 공고 텍스트의 다역할성에 기인하였다.”

“따라서 단일 레이블 강제보다 Top-k 기반 정렬이 실제 채용공고-표준 역량체계 정렬에서 실용적임을 시사한다.”
```

## 📌 NCS–직무 정렬 오분류 유형 분류 프레임워크 (Pilot v1)
🔎 목적

Top-1 및 Top-3 오분류 사례를 구조적으로 분석하여
NCS 세분류 정렬의 한계를 유형화한다.

| 유형 코드 | 유형명                           | 정의                              | 판별 기준                         | 예시 상황                |
| ----- | ----------------------------- | ------------------------------- | ----------------------------- | -------------------- |
| T1    | 의미 중첩형 (Semantic Overlap)     | Gold와 예측이 동일 소분류(인공지능) 내 다른 세분류 | 두 세분류 설명 텍스트 유사도 높음 (≥0.7 권장) | 생성형AI엔지니어링 ↔ 인공지능모델링 |
| T2    | 다역할 혼합형 (Multi-role Blending) | 공고가 2개 이상 능력단위 기능 포함            | 공고 text에 모델링+데이터+배포 등 복합 등장   | 모델 개발 + 서비스 구현       |
| T3    | 키워드 부재형 (Keyword Absence)     | Gold 기준 핵심 키워드가 공고에 거의 없음       | 예: LLM, 프롬프트, RAG 등이 0~1회 등장  | 생성형인데 LLM 언급 없음      |
| T4    | 계층 간 혼선형 (Hierarchical Drift) | 예측이 다른 소분류 또는 중분류로 이동           | 인공지능 → 정보기술운영 등               | 플랫폼 중심 공고            |
| T5    | Gold 라벨 모호형 (Gold Ambiguity)  | 임시 Gold가 과상위 개념으로 지정됨           | 실제 내용은 하위 세분류 특화              | 대부분 생성형으로 찍힌 경우      |


In [12]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# =========================
# 설정
# =========================
NCS_PATH = "datas/processed/ncs_roles_processed.csv"   # role_text, job_role_name 포함
OUT_DIR  = "outputs"
MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

BATCH_SIZE = 64
THRESHOLDS = [0.70, 0.80]   # 논문에서 “중첩 가능” 기준으로 자주 쓰는 구간

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 1) NCS 로드 및 컬럼 확인
# =========================
ncs = pd.read_csv(NCS_PATH)

# 필수 컬럼 자동 선택
NAME_COL = "job_role_name" if "job_role_name" in ncs.columns else None
TEXT_COL = "role_text" if "role_text" in ncs.columns else None

if NAME_COL is None:
    raise ValueError("NCS 파일에 'job_role_name' 컬럼이 필요합니다.")
if TEXT_COL is None:
    raise ValueError("NCS 파일에 'role_text' 컬럼이 필요합니다.")

ncs[NAME_COL] = ncs[NAME_COL].astype(str).str.strip()
ncs[TEXT_COL] = ncs[TEXT_COL].fillna("").astype(str).str.strip()

# 빈 텍스트 제거(안전)
ncs = ncs[ncs[TEXT_COL] != ""].reset_index(drop=True)

labels = ncs[NAME_COL].tolist()
texts  = ncs[TEXT_COL].tolist()

print(f"[INFO] roles: {len(labels)}")

# =========================
# 2) 임베딩 생성
# =========================
model = SentenceTransformer(MODEL_NAME)

emb = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # cosine sim = dot product
)

# =========================
# 3) 세분류 간 유사도 행렬 계산
# =========================
sim = emb @ emb.T  # (N, N)

# 대각선(자기 자신) = 1.0 이므로 제외용 마스크
np.fill_diagonal(sim, np.nan)

# =========================
# 4) 유사도 행렬 저장 (전체)
#    - N이 크면 파일이 커질 수 있음
# =========================
sim_df = pd.DataFrame(sim, index=labels, columns=labels)
sim_matrix_path = os.path.join(OUT_DIR, "role_similarity_matrix.csv")
sim_df.to_csv(sim_matrix_path, encoding="utf-8-sig")
print(f"[OK] saved similarity matrix: {sim_matrix_path}")

# =========================
# 5) Top-overlap pair 추출
#    - (A,B)와 (B,A) 중복 제거 (상삼각만 사용)
# =========================
pairs = []
N = sim.shape[0]

for i in range(N):
    for j in range(i + 1, N):  # i<j => 중복 제거
        s = sim[i, j]
        if np.isnan(s):
            continue
        pairs.append((labels[i], labels[j], float(s)))

pairs_df = pd.DataFrame(pairs, columns=["role_a", "role_b", "cosine_sim"])
pairs_df = pairs_df.sort_values("cosine_sim", ascending=False).reset_index(drop=True)

pairs_all_path = os.path.join(OUT_DIR, "role_overlap_pairs_all.csv")
pairs_df.to_csv(pairs_all_path, index=False, encoding="utf-8-sig")
print(f"[OK] saved all pairs: {pairs_all_path}")

# =========================
# 6) 임계값별 중첩 리스트 저장 + 요약 출력
# =========================
for th in THRESHOLDS:
    overlapped = pairs_df[pairs_df["cosine_sim"] >= th].copy()
    out_path = os.path.join(OUT_DIR, f"role_overlap_pairs_ge_{int(th*100)}.csv")
    overlapped.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n[SUMMARY] cosine_sim >= {th:.2f}")
    print(f"- #pairs: {len(overlapped)}")
    print("- top 10:")
    print(overlapped.head(10).to_string(index=False))

    print(f"[OK] saved: {out_path}")

# =========================
# 7) (선택) 각 role별 “가장 겹치는 상대” Top-1 / Top-3 요약
# =========================
topk_rows = []
TOPK = 3

for i in range(N):
    row = sim[i].copy()  # nan 포함
    # 가장 큰 유사도 TOPK 인덱스
    idx = np.argsort(-np.nan_to_num(row, nan=-1.0))[:TOPK]
    for rank, j in enumerate(idx, start=1):
        if np.isnan(sim[i, j]):
            continue
        topk_rows.append({
            "role": labels[i],
            "rank": rank,
            "neighbor_role": labels[j],
            "cosine_sim": float(sim[i, j])
        })

topk_df = pd.DataFrame(topk_rows).sort_values(["role", "rank"]).reset_index(drop=True)
topk_path = os.path.join(OUT_DIR, f"role_top{TOPK}_neighbors.csv")
topk_df.to_csv(topk_path, index=False, encoding="utf-8-sig")
print(f"\n[OK] saved top-{TOPK} neighbors per role: {topk_path}")

[INFO] roles: 7


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] saved similarity matrix: outputs\role_similarity_matrix.csv
[OK] saved all pairs: outputs\role_overlap_pairs_all.csv

[SUMMARY] cosine_sim >= 0.70
- #pairs: 20
- top 10:
     role_a      role_b  cosine_sim
  인공지능서비스기획   인공지능서비스구현    0.848415
    인공지능모델링   인공지능서비스구현    0.845194
  인공지능플랫폼구축   인공지능서비스기획    0.836126
  인공지능서비스기획 인공지능서비스운영관리    0.832768
    인공지능모델링 인공지능학습데이터구축    0.818454
  인공지능서비스기획     인공지능모델링    0.815406
  인공지능플랫폼구축   인공지능서비스구현    0.807399
인공지능서비스운영관리   인공지능서비스구현    0.782120
  인공지능플랫폼구축 인공지능서비스운영관리    0.780560
  인공지능서비스구현 인공지능학습데이터구축    0.759054
[OK] saved: outputs\role_overlap_pairs_ge_70.csv

[SUMMARY] cosine_sim >= 0.80
- #pairs: 7
- top 10:
   role_a      role_b  cosine_sim
인공지능서비스기획   인공지능서비스구현    0.848415
  인공지능모델링   인공지능서비스구현    0.845194
인공지능플랫폼구축   인공지능서비스기획    0.836126
인공지능서비스기획 인공지능서비스운영관리    0.832768
  인공지능모델링 인공지능학습데이터구축    0.818454
인공지능서비스기획     인공지능모델링    0.815406
인공지능플랫폼구축   인공지능서비스구현    0.807399
[OK] saved: outputs\role_overlap_pairs_ge_80.csv

[OK] s

In [14]:
# 우리는 묻습니다:

# 세분류 A와 세분류 B의 role_text는
# 실제로 얼마나 유사한가?

In [13]:
# 만약 cosine similarity가:
# 0.8 이상이면 → 거의 같은 의미 영역
# 0.7 이상이면 → 경계가 모호

In [15]:
# 📌 이게 왜 논문적으로 중요한가?

# 정확도만 보고

# “SBERT 성능이 낮다”

# 라고 하면 논문이 약합니다.

# 하지만 이렇게 말하면 다릅니다:

# 세분류 간 평균 유사도가 0.75 이상으로 높게 나타났으며,
# 이는 NCS 세분류 간 의미 경계가 구조적으로 중첩됨을 시사한다.
# 따라서 단일 레이블 강제 정렬은 구조적 한계를 가진다.

# 이건 완전히 다른 수준입니다.

In [16]:
# 🔬 즉, 이 분석의 목적은

# ✔ 모델을 평가하는 것이 아니라
# ✔ NCS 체계의 “구조적 분리 가능성”을 검증하는 것

# 입니다.

In [17]:
# 🎓 박사 논문 관점에서의 의미

# 이 분석은 다음을 가능하게 합니다:

# Top-1이 낮은 이유를 구조적으로 설명

# Top-k 정렬의 타당성 근거 제공

# “의미 정렬 문제”를 문제 정의로 격상

# 단순 모델 비교 논문에서 벗어남

In [19]:
# 만약 세분류 간 유사도가 전부 0.4~0.5 수준으로 낮게 나오면,
# 그때는 무엇을 의미할까요?

# 그 의미는 매우 명확합니다.

# ✅ NCS 세분류들은 의미적으로 충분히 분리되어 있다.
# ❌ 구조적 중첩 문제는 아니다.

# 즉,
# Top-1 정확도가 낮은 이유는 **NCS 구조 문제가 아니라 “정렬 설계 문제”**입니다.

In [20]:
# 🔎 그럼 해석이 완전히 달라집니다

# 현재 우리는 두 가지 가설 중 하나를 검증하고 있습니다.

# 가설 A
# NCS 세분류는 의미적으로 겹쳐 있다 → 정렬이 구조적으로 어렵다

# 가설 B
# NCS 세분류는 충분히 분리되어 있다 → 모델/텍스트/설계 문제다

# 유사도가 0.4~0.5라면
# 가설 A는 기각되고, 가설 B가 채택됩니다.

🔥 전략적 해석

| 세분류 유사도 | 논문 방향            |
| ------- | ---------------- |
| 0.7~0.9 | NCS 구조 중첩 문제 제기  |
| 0.5~0.7 | 부분 중첩 + 설계 문제 혼합 |
| 0.4~0.5 | 정렬 방법론 문제 중심     |


🧠 핵심

유사도가 낮으면: 
- 모델 개선 방향으로 
- role_text 재설계
- 스킬 추출 후 매핑
- 계층 단계별 매핑
으로 갑니다.

유사도가 높으면:
- 구조 중첩 문제 제기
- Top-k 기반 정렬 필요성 주장